<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Algorithmic Trading

**Chapter 11 &mdash; Trading Cryptocurrencies with Gemini**

## Introduction

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_core.git
import sys
sys.path.append('python_for_algo_trading_core')


In [ ]:
%matplotlib inline
from pylab import mpl, plt
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'

In [ ]:
import quandl as q
import configparser as cp

In [ ]:
c = cp.ConfigParser()
c.read('../../../data/pyalgo.cfg')
q.ApiConfig.api_key = c['quandl']['api_key']

In [ ]:
bn = q.get('BCHAIN/TOTBC') / 1e6

In [ ]:
bn.plot(figsize=(10, 6));

In [ ]:
be = q.get('BCHAIN/MKPRU')

In [ ]:
be.plot(figsize=(10, 6));

In [ ]:
bm = q.get('BCHAIN/MKTCP') / 1e9

In [ ]:
bm.plot(figsize=(10, 6));

## A Wrapper Class for the Gemini API

In [ ]:
import tpqge

In [ ]:
con = tpqge.tpqge(c['gemini']['key'],
                  c['gemini']['secret_key'])

In [ ]:
symbols = con.get_symbols()

In [ ]:
symbols

In [ ]:
con.get_ticker('btcusd')

In [ ]:
con.get_ticker('ethusd')

In [ ]:
ticker = con.get_ticker('ethbtc')

In [ ]:
ticker

In [ ]:
import datetime as dt

In [ ]:
dt.datetime.fromtimestamp(int(ticker['volume']['timestamp']) /
                          1e3)

## Retrieving Historical Data

In [ ]:
data = con.get_trades_history('btcusd',
                    since=dt.datetime(2020, 9, 27, 8, 0, 0),
                    limit_trades=500,
                    include_breaks=False)

In [ ]:
data.info()

In [ ]:
data.round(3).tail()

In [ ]:
data['amount'].hist(bins=35);

In [ ]:
data[['price', 'amount']].plot(secondary_y='amount',
                               figsize=(10, 6));

In [ ]:
order = con.get_current_order_book('btcusd',
                                   limit_asks=5,
                                   limit_bids=5)

In [ ]:
import pprint

In [ ]:
pprint.pprint(order)

In [ ]:
order['asks'][0]['timestamp']

In [ ]:
dt.datetime.fromtimestamp(int(order['asks'][0]['timestamp']))

In [ ]:
import pandas as pd

In [ ]:
order = con.get_current_order_book('ethusd',
                                   limit_asks=7,
                                   limit_bids=7)

In [ ]:
bids = pd.DataFrame(order['bids'])

In [ ]:
bids

In [ ]:
bids['price'].astype(float).hist(bins=50, figsize=(10, 6));

## Placing and Managing Orders via the API

In [ ]:
new_order_result_1 = con.new_order(symbol='btcusd',
                                   amount=0.001,
                                   price=11250,
                                   side='buy')

In [ ]:
new_order_result_1

In [ ]:
new_order_result_2 = con.new_order(symbol='btcusd',
                                   amount=0.0001,
                                   price=10750,
                                   side='buy')

In [ ]:
new_order_result_2

In [ ]:
order_id = new_order_result_2['order_id']

In [ ]:
status = con.get_order_status(order_id)

In [ ]:
status

In [ ]:
orders = con.get_active_orders()
orders

In [ ]:
# cancel = con.cancel_order(order_id)

In [ ]:
# cancel

In [ ]:
cancel = con.cancel_all_session_orders()

## Most Recent Transaction History

In [ ]:
con = tpqge.tpqge(c['gemini']['key'],
                  c['gemini']['secret_key'])

In [ ]:
since = dt.datetime.now() - dt.timedelta(hours=4)

In [ ]:
trades = con.get_trades_history('btcusd',
                        since=since,
                        limit_trades=100,
                        include_breaks=False)
trades

In [ ]:
trades = con.get_trades_history('btcusd', since=since,
                                limit_trades=500,
                                include_breaks=False)
print(40 * '-')
print('timestamp %s' % dt.datetime.fromtimestamp(
                        float(trades['timestamp'].iloc[-1])))
print('#   trades: %d' % len(trades))
print('ave. price: %f' % float(trades['price'].mean()))
print('min  price: %f' % float(trades['price'].min()))
print('max  price: %f' % float(trades['price'].max()))
print('last price: %f' % float(trades['price'].iloc[-1]))
print(40 * '-')

## Implementing Trading Strategies in Real-Time 

### Simple Example

In [ ]:
import json

In [ ]:
import pandas as pd

In [ ]:
data = pd.DataFrame()

In [ ]:
messages = []

In [ ]:
def on_message(ws, message):
    global data
    global resam
    timestamp = dt.datetime.now()
    msg = json.loads(message)
    try:
        side = msg['events'][0]['side']
        price = float(msg['events'][0]['price'])
        if msg['type'] == 'update':
            print('%s | %s price = %f' % (timestamp,
                                        side, price))
            data = data.append(pd.DataFrame({side: price},
                                            index=[timestamp]),
                              sort=True)
            messages.append(message)
            resam = data.resample('5s',
                        label='right').last().ffill()
            resam['mid'] = resam.mean(axis=1)
    except:
        pass

In [ ]:
import websocket

In [ ]:
ws = websocket.WebSocketApp(
    'wss://api.gemini.com/v1/marketdata/BTCUSD',
     on_message=on_message)

In [ ]:
ws.run_forever(ping_interval=5)

In [ ]:
data.tail()

In [ ]:
resam.tail()

### Using the Python Wrapper Class

#### The market data stream


In [ ]:
con = tpqge.tpqge(c['gemini']['key'],
                  c['gemini']['secret_key'])

In [ ]:
stream = con.make_market_data_streamer('btcusd')
stream

In [ ]:
stream.start_streaming()

In [ ]:
stream.is_alive()

In [ ]:
stream.stop_streaming()

In [ ]:
stream.is_alive()

#### Callbacks

In [ ]:
import time

In [ ]:
def example_callback(data):
    print('Received an event of type %s.' % data['type'].upper())
    print('Its subtype is %s.' % data['reason'].upper())
    print('Available field names are: ')
    print(list(data.keys()))
    print('price is %f\n' % float(data['price']))
    # time.sleep(1)

In [ ]:
stream.set_callback('change', example_callback)
stream.set_callback('place_change', example_callback)

In [ ]:
stream.start_streaming()

In [ ]:
stream.is_alive()

In [ ]:
stream.stop_streaming()

#### Order books

In [ ]:
import time

In [ ]:
con2 = tpqge.tpqge(c['gemini']['key'],
                   c['gemini']['secret_key'])

In [ ]:
con2.get_order_book('btcusd', 'ask')

In [ ]:
stream2 = con2.make_market_data_streamer('btcusd')

In [ ]:
stream2.start_streaming()
time.sleep(5)
stream2.stop_streaming()

In [ ]:
con2.get_order_book('btcusd', 'ask')

#### The order status stream

In [ ]:
con3 = tpqge.tpqge(c['gemini']['key'],
                   c['gemini']['secret_key'])

In [ ]:
oss = con3.make_order_status_streamer()
oss

In [ ]:
oss.start_streaming()

In [ ]:
oss.stop_streaming()

In [ ]:
oss.is_alive()

## Retrieving Account Information

In [ ]:
con = tpqge.tpqge(c['gemini']['key'],
                  c['gemini']['secret_key'])

In [ ]:
balances = con.get_available_balances()

In [ ]:
balances

In [ ]:
volumes = con.get_trade_volume()

In [ ]:
volumes

In [ ]:
orders = con.get_active_orders()

In [ ]:
orders

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>